# TP1 (suite) — Émotions faciales sur CK+ : HOG, LBP et landmarks du visage

**Objectif.** Reconnaître 7 émotions à partir de photos de visages (CK+), puis aller plus loin : construire un modèle qui **mesure des distances sur le visage et prédit l'émotion**, utilisable plus tard avec une **caméra en direct**.

**Trois façons de « regarder » un visage**
- **HOG** : les contours (forme).
- **LBP** : la micro-texture (plis, détails).
- **Landmarks** : 68 points repérés sur le visage (yeux, sourcils, nez, bouche, mâchoire) → distances, angles, courbures.

**Règle du jeu.** Le test est toujours fait sur des **personnes jamais vues** à l'entraînement (découpage par sujet).

| Partie | Contenu |
|---|---|
| 0 | Installation et code partagé |
| 1–2 | Données, découpage par sujet, déséquilibre |
| 3–4 | HOG, LBP |
| 5–6 | Fusion HOG+LBP, sélection de caractéristiques |
| 7 | Bilan du TP1 |
| 8 | **Suite : état de l'art, landmarks, fusion** |
| 9 | Comparaison finale |
| 10 | **Modèle final + caméra en direct** |

> **Durée.** Environ 20 à 40 minutes. Mettre `QUICK = True` plus bas pour un essai rapide.

# Partie 0 — Installation et code partagé

In [ ]:
import sys, subprocess, importlib, warnings

warnings.filterwarnings("ignore")


def installer(pip_name, import_name=None):
    try:
        importlib.import_module(import_name or pip_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])


installer("kagglehub")
installer("scikit-image", "skimage")
installer("opencv-python-headless", "cv2")
installer("xgboost")
try:
    installer("dlib-bin", "dlib")
except Exception:
    installer("dlib")

### Code partagé : `emotion_landmarks.py`

La cellule suivante écrit un petit fichier Python qui contient **toute la chaîne de calcul** : visage → 68 landmarks → mesures → patchs HOG/LBP. Le notebook l'utilise pour s'entraîner, et la **caméra** l'utilisera pour prédire : ainsi, entraînement et direct calculent *exactement* les mêmes choses.

Inutile de le lire maintenant : chaque fonction est expliquée à l'endroit où elle sert (Partie 8).

In [ ]:
%%writefile emotion_landmarks.py
"""
emotion_landmarks.py
Code partage par le notebook (entrainement) et par la camera (prediction en direct).

Chaine complete :  image -> visage 48x48 -> 68 landmarks (dlib)
                   -> mesures geometriques (distances, angles, courbures...)
                   -> patchs HOG+LBP autour des yeux / nez / bouche
                   -> classifieur SVM -> emotion
"""
import bz2
import urllib.request
from pathlib import Path

import cv2
import numpy as np
from skimage.feature import hog, local_binary_pattern
from sklearn.base import BaseEstimator, TransformerMixin

IMG = 48      # taille des visages CK+ (48 x 48 pixels)
UP = 192      # taille d'agrandissement avant la detection des landmarks
PREDICTOR_FILE = "shape_predictor_68_face_landmarks.dat"


# ----------------------------------------------------------------------------
# 1) Mise a l'echelle par blocs (chaque famille de caracteristiques pese pareil)
# ----------------------------------------------------------------------------
class BlockScaler(BaseEstimator, TransformerMixin):
    """Standardise chaque colonne, puis divise chaque bloc par racine(taille du bloc).
    Ainsi un bloc de 3000 colonnes (HOG) ne ecrase pas un bloc de 80 colonnes (geometrie)."""

    def __init__(self, blocks=None):
        self.blocks = blocks          # liste de (debut, fin) ; None = un seul bloc

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0) + 1e-8
        self.weights_ = np.ones(X.shape[1])
        for debut, fin in (self.blocks or [(0, X.shape[1])]):
            self.weights_[debut:fin] = 1.0 / np.sqrt(fin - debut)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return (X - self.mean_) / self.std_ * self.weights_


# ----------------------------------------------------------------------------
# 2) LBP : histogrammes par cellules
# ----------------------------------------------------------------------------
def lbp_histogram(img, P=8, R=1, grid=(1, 1)):
    code = local_binary_pattern(img, P, R, "uniform").astype(int)
    n_bins = P + 2
    h, w = code.shape
    parts = []
    for i in range(grid[0]):
        for j in range(grid[1]):
            cell = code[i * h // grid[0]:(i + 1) * h // grid[0], j * w // grid[1]:(j + 1) * w // grid[1]]
            hist = np.bincount(cell.ravel(), minlength=n_bins)[:n_bins].astype(np.float32)
            parts.append(np.sqrt(hist / (hist.sum() + 1e-8)))     # racine = distance de Hellinger
    return np.concatenate(parts)


# ----------------------------------------------------------------------------
# 3) Detection des 68 landmarks avec dlib
# ----------------------------------------------------------------------------
def load_dlib(predictor_file=PREDICTOR_FILE):
    """Retourne (detecteur de visages, fonction de landmarks dlib).
    La fonction de landmarks prend une image grise 192x192 et rend un tableau (68, 2) en pixels."""
    import dlib
    path = Path(predictor_file)
    if not path.exists():
        print("Telechargement du modele dlib (environ 100 Mo)...")
        url = "http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2"
        path.write_bytes(bz2.decompress(urllib.request.urlopen(url, timeout=120).read()))
    forme = dlib.shape_predictor(str(path))
    boite = dlib.rectangle(int(0.02 * UP), int(0.02 * UP), int(0.98 * UP) - 1, int(0.98 * UP) - 1)

    def predire(image):
        shape = forme(image, boite)
        return np.array([[p.x, p.y] for p in shape.parts()], dtype=float)

    return dlib.get_frontal_face_detector(), predire


def landmarks_from_crop(gray48, predictor, n_views=4):
    """68 points (x, y) d'un visage 48x48. On moyenne quelques vues legerement
    decalees / tournees : le resultat est plus stable qu'une seule detection."""
    big = cv2.resize(gray48, (UP, UP), interpolation=cv2.INTER_CUBIC)
    rng = np.random.default_rng(0)
    centre = np.array([UP / 2, UP / 2])
    all_points = []
    for vue in range(n_views):
        if vue == 0:
            A = np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]])
        else:
            angle = np.deg2rad(rng.uniform(-6, 6))
            echelle = 1 + rng.uniform(-0.05, 0.05)
            decalage = rng.uniform(-0.04, 0.04, 2) * UP
            A = np.array([[echelle * np.cos(angle), -echelle * np.sin(angle), 0.0],
                          [echelle * np.sin(angle), echelle * np.cos(angle), 0.0]])
            A[:, 2] = centre + decalage - A[:, :2] @ centre
        image = cv2.warpAffine(big, A, (UP, UP), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
        points = predictor(image)                       # (68, 2) en pixels de l'image 192x192
        A_inv = cv2.invertAffineTransform(A)
        all_points.append(points @ A_inv[:, :2].T + A_inv[:, 2])     # retour dans l'image d'origine
    return np.mean(all_points, axis=0) * IMG / UP


def align_landmarks(L):
    """L : (n, 68, 2). Centre sur le milieu des yeux, tourne pour que les yeux soient
    horizontaux, et prend la distance entre les yeux (IOD) comme unite de longueur."""
    oeil_D = L[:, 36:42].mean(axis=1)
    oeil_G = L[:, 42:48].mean(axis=1)
    d = oeil_G - oeil_D
    iod = np.linalg.norm(d, axis=1)
    angle = np.arctan2(d[:, 1], d[:, 0])
    c, s = np.cos(-angle)[:, None], np.sin(-angle)[:, None]
    centre = L - ((oeil_D + oeil_G) / 2)[:, None, :]
    x = centre[..., 0] * c - centre[..., 1] * s
    y = centre[..., 0] * s + centre[..., 1] * c
    return np.stack([x, y], axis=-1) / iod[:, None, None], iod


# ----------------------------------------------------------------------------
# 4) Mesures geometriques (indices = schema standard a 68 points)
#    D = cote droit de la personne, G = cote gauche
# ----------------------------------------------------------------------------
DISTANCES = {
    "sourcils_ecart": (21, 22), "sourcil_D_int_oeil": (21, 39), "sourcil_G_int_oeil": (22, 42),
    "sourcil_D_milieu_oeil": (19, 37), "sourcil_G_milieu_oeil": (24, 44),
    "sourcil_D_ext_oeil": (17, 36), "sourcil_G_ext_oeil": (26, 45),
    "nez_longueur": (27, 30), "nez_largeur": (31, 35), "nez_levre": (33, 51), "bout_nez_levre": (30, 51),
    "bouche_largeur": (48, 54), "bouche_hauteur_ext": (51, 57), "bouche_ouverture": (62, 66),
    "bouche_largeur_int": (60, 64), "levre_sup_epaisseur": (51, 62), "levre_inf_epaisseur": (57, 66),
    "coin_bouche_D_nez": (48, 33), "coin_bouche_G_nez": (54, 33),
    "coin_bouche_D_oeil": (48, 36), "coin_bouche_G_oeil": (54, 45),
    "menton_levre": (8, 57), "menton_nez": (8, 33), "machoire_largeur": (4, 12),
    "oeil_D_largeur": (36, 39), "oeil_G_largeur": (42, 45),
}
ANGLES = {   # angle au point du milieu
    "coin_bouche_D": (49, 48, 59), "coin_bouche_G": (53, 54, 55), "arc_levre_sup": (50, 51, 52),
    "arche_sourcil_D": (17, 19, 21), "arche_sourcil_G": (22, 24, 26),
    "coin_oeil_D": (37, 36, 41), "coin_oeil_G": (44, 45, 46),
    "menton": (6, 8, 10), "base_nez": (31, 33, 35), "triangle_bouche": (48, 51, 54),
}
COURBES = {
    "sourcil_D": range(17, 22), "sourcil_G": range(22, 27), "machoire": range(0, 17),
    "levre_sup": range(48, 55), "levre_inf": [54, 55, 56, 57, 58, 59, 48], "bas_nez": range(31, 36),
}
CONTOURS = {"oeil_D": range(36, 42), "oeil_G": range(42, 48), "bouche_ext": range(48, 60), "bouche_int": range(60, 68)}
TRIANGLES = [(17, 19, 21), (22, 24, 26), (36, 37, 41), (39, 38, 40), (42, 43, 47), (45, 44, 46), (21, 39, 27),
             (22, 42, 27), (31, 33, 35), (48, 51, 54), (48, 57, 54), (48, 33, 54), (36, 48, 8), (45, 54, 8)]

# mesures les plus parlantes, affichees par la camera
MESURES_AFFICHEES = ["d_bouche_largeur", "d_bouche_ouverture", "ouverture_bouche", "ouverture_oeil_D",
                     "ouverture_oeil_G", "d_sourcil_D_milieu_oeil", "d_sourcil_G_milieu_oeil", "commissures_relevees"]


def _dist(L, a, b):
    return np.linalg.norm(L[:, a] - L[:, b], axis=1)


def _angle(L, a, b, c):
    v1, v2 = L[:, a] - L[:, b], L[:, c] - L[:, b]
    cos = (v1 * v2).sum(axis=1) / (np.linalg.norm(v1, axis=1) * np.linalg.norm(v2, axis=1) + 1e-9)
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))


def _aire_triangle(L, a, b, c):
    u, v = L[:, b] - L[:, a], L[:, c] - L[:, a]
    return 0.5 * np.abs(u[:, 0] * v[:, 1] - u[:, 1] * v[:, 0])


def _contour(L, indices):
    """Aire et circularite d'un contour ferme."""
    P = L[:, list(indices)]
    Q = np.roll(P, -1, axis=1)
    aire = 0.5 * np.abs((P[..., 0] * Q[..., 1] - P[..., 1] * Q[..., 0]).sum(axis=1))
    perimetre = np.linalg.norm(P - Q, axis=2).sum(axis=1)
    return aire, 4 * np.pi * aire / (perimetre ** 2 + 1e-9)


def _courbure(L, indices):
    """Parabole y = a x^2 + b x + c passant au mieux par une suite de points ; on garde (a, b)."""
    idx = list(indices)
    sortie = np.zeros((len(L), 2))
    for i in range(len(L)):
        sortie[i] = np.polyfit(L[i, idx, 0], L[i, idx, 1], 2)[:2]
    return sortie


def geometry_features(L):
    """L : landmarks alignes (n, 68, 2). Retourne (tableau (n, 80), liste des 80 noms)."""
    valeurs, noms = [], []

    def ajouter(nom, v):
        noms.append(nom)
        valeurs.append(np.asarray(v, dtype=float))

    for nom, (a, b) in DISTANCES.items():
        ajouter("d_" + nom, _dist(L, a, b))
    ajouter("ouverture_oeil_D", (_dist(L, 37, 41) + _dist(L, 38, 40)) / (2 * _dist(L, 36, 39) + 1e-9))
    ajouter("ouverture_oeil_G", (_dist(L, 43, 47) + _dist(L, 44, 46)) / (2 * _dist(L, 42, 45) + 1e-9))
    ajouter("ouverture_bouche", (_dist(L, 61, 67) + _dist(L, 62, 66) + _dist(L, 63, 65)) / (3 * _dist(L, 60, 64) + 1e-9))
    milieu_bouche_y = (L[:, 51, 1] + L[:, 57, 1]) / 2          # un coin releve = sourire
    ajouter("coin_releve_D", milieu_bouche_y - L[:, 48, 1])
    ajouter("coin_releve_G", milieu_bouche_y - L[:, 54, 1])
    ajouter("commissures_relevees", milieu_bouche_y - (L[:, 48, 1] + L[:, 54, 1]) / 2)
    ajouter("sourcil_D_haut", L[:, 36, 1] - L[:, 19, 1])
    ajouter("sourcil_G_haut", L[:, 45, 1] - L[:, 24, 1])
    for nom, (a, b, c) in ANGLES.items():
        ajouter("angle_" + nom, _angle(L, a, b, c))
    pente = lambda a, b: np.degrees(np.arctan2(L[:, b, 1] - L[:, a, 1], L[:, b, 0] - L[:, a, 0]))
    ajouter("pente_sourcil_D", pente(17, 21))
    ajouter("pente_sourcil_G", -pente(26, 22))
    for nom, indices in COURBES.items():
        a, b = _courbure(L, indices).T
        ajouter("courbe_" + nom + "_a", a)
        ajouter("courbe_" + nom + "_b", b)
    for nom, indices in CONTOURS.items():
        aire, circ = _contour(L, indices)
        ajouter("aire_" + nom, aire)
        ajouter("circularite_" + nom, circ)
    for k, (a, b, c) in enumerate(TRIANGLES):
        ajouter("triangle_%d" % k, _aire_triangle(L, a, b, c))
    return np.nan_to_num(np.stack(valeurs, axis=1)), noms


# ----------------------------------------------------------------------------
# 5) Patchs d'apparence (HOG + LBP) autour des yeux, du nez et de la bouche
# ----------------------------------------------------------------------------
REGIONS = {   # indices des landmarks de la zone, taille du patch (hauteur, largeur)
    "yeux_sourcils": (list(range(17, 27)) + list(range(36, 48)), (24, 56)),
    "nez": (list(range(27, 36)), (24, 24)),
    "bouche": (list(range(48, 68)), (24, 40)),
}


def decouper_zone(img, landmarks, indices, taille, marge=0.2):
    pts = landmarks[indices]
    x0, y0 = pts.min(axis=0)
    x1, y1 = pts.max(axis=0)
    w, h = max(x1 - x0, 6), max(y1 - y0, 6)
    xa = int(max(0, np.floor(x0 - marge * w)))
    xb = int(min(img.shape[1], np.ceil(x1 + marge * w)))
    ya = int(max(0, np.floor(y0 - marge * h)))
    yb = int(min(img.shape[0], np.ceil(y1 + marge * h)))
    if xb - xa < 4:
        xa, xb = max(0, xa - 2), min(img.shape[1], xb + 2)
    if yb - ya < 4:
        ya, yb = max(0, ya - 2), min(img.shape[0], yb + 2)
    return cv2.resize(img[ya:yb, xa:xb], (taille[1], taille[0]), interpolation=cv2.INTER_CUBIC)


def patch_features(gray48, landmarks):
    sortie = []
    for indices, taille in REGIONS.values():
        patch = decouper_zone(gray48, landmarks, indices, taille)
        sortie.append(hog(patch, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2),
                          block_norm="L2-Hys", feature_vector=True))
        sortie.append(lbp_histogram(patch, 8, 1, (2, 2)))
    return np.concatenate(sortie)


def face_features(gray48, predictor, n_views=4):
    """Vecteur d'un visage 48x48 = [80 mesures geometriques | patchs HOG+LBP].
    Retourne aussi les landmarks (dans l'image 48x48) et le dictionnaire des mesures."""
    landmarks = landmarks_from_crop(gray48, predictor, n_views)
    alignes, _ = align_landmarks(landmarks[None])
    geo, noms = geometry_features(alignes)
    vecteur = np.concatenate([geo[0], patch_features(gray48, landmarks)])
    return vecteur, landmarks, dict(zip(noms, geo[0]))


# ----------------------------------------------------------------------------
# 6) Cadrage : comment decouper un visage de camera pour qu'il ressemble a CK+
# ----------------------------------------------------------------------------
def measure_framing(images48, detector, n_max=300):
    """Dans les images CK+, quelle part de l'image le visage detecte par dlib occupe-t-il ?
    On s'en sert ensuite pour decouper les visages de la camera de la meme facon."""
    rapports, decalages = [], []
    for img in images48[:n_max]:
        grand = cv2.resize(img, (UP, UP), interpolation=cv2.INTER_CUBIC)
        faces = detector(grand, 0)
        if len(faces) == 0:
            continue
        f = max(faces, key=lambda r: r.width())
        rapports.append(f.width() / UP)
        decalages.append(((f.top() + f.bottom()) / 2 - UP / 2) / UP)
    taux = len(rapports) / max(1, min(n_max, len(images48)))
    if len(rapports) < 10:
        return (0.65, 0.0), taux            # valeurs par defaut si dlib detecte trop peu de visages
    return (float(np.median(rapports)), float(np.median(decalages))), taux


def crop_face(gray, box, framing):
    """Decoupe un carre autour du visage detecte et le redimensionne en 48x48.
    Retourne (visage48, x0, y0, cote) pour pouvoir remettre les landmarks dans l'image."""
    ratio, dy = framing
    cote = box.width() / ratio
    cx = (box.left() + box.right()) / 2
    cy = (box.top() + box.bottom()) / 2 - dy * cote
    s = max(int(round(cote)), 8)
    x0, y0 = int(round(cx - cote / 2)), int(round(cy - cote / 2))
    marge = s
    cadre = cv2.copyMakeBorder(gray, marge, marge, marge, marge, cv2.BORDER_REPLICATE)
    xa, ya = max(x0 + marge, 0), max(y0 + marge, 0)
    morceau = cadre[ya:ya + s, xa:xa + s]
    return cv2.resize(morceau, (IMG, IMG), interpolation=cv2.INTER_AREA), x0, y0, s


# ----------------------------------------------------------------------------
# 7) Analyseur : image de camera -> visage, landmarks, mesures, emotion
# ----------------------------------------------------------------------------
class EmotionAnalyzer:
    def __init__(self, model_file="emotion_model.joblib", min_confidence=0.5):
        import joblib
        bundle = joblib.load(model_file)
        self.pipeline = bundle["pipeline"]
        self.classes = list(bundle["classes"])
        self.framing = bundle["framing"]
        self.n_views = bundle["n_views"]
        self.geo_only = bundle.get("geo_only", False)      # True si le modele n'utilise que les mesures geometriques
        self.min_confidence = min_confidence
        self.detector, self.predictor = load_dlib()

    def predict_crop(self, gray48):
        """Predit a partir d'un visage deja decoupe en 48x48."""
        vecteur, landmarks, mesures = face_features(gray48, self.predictor, self.n_views)
        if self.geo_only:
            vecteur = vecteur[:len(mesures)]
        proba = self.pipeline.predict_proba(vecteur[None])[0]
        meilleur = int(np.argmax(proba))
        emotion = self.classes[meilleur]
        confiance = float(proba[meilleur])
        return {"emotion": emotion, "confidence": confiance,
                "label": emotion if confiance >= self.min_confidence else "incertain",
                "probabilities": dict(zip(self.classes, proba.tolist())),
                "distances": mesures, "landmarks48": landmarks}

    def analyze(self, frame):
        """Image de camera (BGR ou gris). Retourne None si aucun visage, sinon un dictionnaire :
        box (x, y, l, h), landmarks (68 points dans l'image), distances (en unites de
        distance-entre-les-yeux), iod_px (cette distance en pixels), emotion, confidence, label."""
        import dlib
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) if frame.ndim == 3 else frame
        reduction = 480.0 / gray.shape[1] if gray.shape[1] > 480 else 1.0       # detection plus rapide
        petit = cv2.resize(gray, None, fx=reduction, fy=reduction) if reduction < 1 else gray
        faces = self.detector(petit, 0)
        if len(faces) == 0:
            return None
        f = max(faces, key=lambda r: r.width())
        box = dlib.rectangle(int(f.left() / reduction), int(f.top() / reduction),
                             int(f.right() / reduction), int(f.bottom() / reduction))
        visage, x0, y0, cote = crop_face(gray, box, self.framing)
        res = self.predict_crop(visage)
        facteur = cote / IMG
        lm = res.pop("landmarks48") * facteur + np.array([x0, y0])
        yeux_D, yeux_G = lm[36:42].mean(axis=0), lm[42:48].mean(axis=0)
        res.update({"box": (box.left(), box.top(), box.width(), box.height()),
                    "landmarks": lm, "iod_px": float(np.linalg.norm(yeux_G - yeux_D))})
        return res

In [ ]:
import time, re, hashlib, itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from skimage.feature import hog, local_binary_pattern
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

from emotion_landmarks import (BlockScaler, lbp_histogram, load_dlib, face_features, measure_framing,
                               EmotionAnalyzer, IMG, MESURES_AFFICHEES)

# ---------------- réglages
SEED = 42
QUICK = False        # True = essai rapide (moins d'essais pour Random Forest et XGBoost)
N_FOLDS = 5          # validation croisée (par sujet)
N_ITER = 3 if QUICK else 5
N_VIEWS = 4          # nombre de vues moyennées pour détecter les landmarks
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

# Partie 1 — Chargement et exploration de CK+

Le dataset est téléchargé avec `kagglehub`. L'**émotion** est le nom du dossier, le **sujet** (la personne) est lu dans le nom du fichier (`S010_004_...` = sujet `S010`).

In [ ]:
import kagglehub
KAGGLE_DATASET = "shawon10/ckplus"
DATA_ROOT = Path(kagglehub.dataset_download(KAGGLE_DATASET))
print("DATA_ROOT =", DATA_ROOT)

In [ ]:
EMOTIONS = {"anger", "contempt", "disgust", "fear", "happy", "sadness", "surprise", "neutral"}

lignes, images = [], []
illisibles, autre_taille = 0, 0
for chemin in sorted(DATA_ROOT.rglob("*")):
    if chemin.suffix.lower() not in {".png", ".jpg", ".jpeg", ".bmp"}:
        continue
    emotion = chemin.parent.name.lower()
    if emotion not in EMOTIONS:
        continue
    gris = cv2.imread(str(chemin), cv2.IMREAD_GRAYSCALE)
    if gris is None:
        illisibles += 1
        continue
    if gris.shape != (IMG, IMG):
        autre_taille += 1
        gris = cv2.resize(gris, (IMG, IMG), interpolation=cv2.INTER_AREA)
    sujet = re.search(r"(S\d{2,})", chemin.stem, re.I)
    lignes.append({"emotion": emotion, "subject": sujet.group(1).upper() if sujet else None,
                   "md5": hashlib.md5(gris.tobytes()).hexdigest(), "std": float(gris.std())})
    images.append(gris)

df_brut = pd.DataFrame(lignes)
X_brut = np.stack(images)

# ----- contrôle des images et des étiquettes
copies = df_brut.duplicated("md5", keep="first")
nb_emotions_par_image = df_brut.groupby("md5")["emotion"].nunique()
conflits = nb_emotions_par_image[nb_emotions_par_image > 1].index          # même image, étiquettes différentes
print("Images trouvées            :", len(df_brut))
print("Illisibles                 :", illisibles)
print("Taille différente de 48x48 :", autre_taille)
print("Sujet introuvable          :", int(df_brut.subject.isna().sum()))
print("Presque uniformes          :", int((df_brut["std"] < 2).sum()))
print("Copies exactes (mêmes pixels) :", int(copies.sum()), "-> supprimées")
print("Étiquettes contradictoires :", len(conflits))

garder = (~copies) & df_brut.subject.notna() & (df_brut["std"] >= 2) & (~df_brut.md5.isin(conflits))
df = df_brut[garder].reset_index(drop=True)
X_img = X_brut[garder.values]

le = LabelEncoder().fit(df.emotion)
CLASSES = list(le.classes_)
y = le.transform(df.emotion)
groupes = df.subject.values
print("\nImages gardées : %d | sujets : %d | classes : %s" % (len(df), df.subject.nunique(), CLASSES))

In [ ]:
# ----- distribution des classes
tableau = pd.DataFrame({"images": df.emotion.value_counts(), "sujets": df.groupby("emotion").subject.nunique()}).loc[CLASSES]
tableau["images par sujet"] = (tableau["images"] / tableau["sujets"]).round(1)
tableau["part (%)"] = (100 * tableau["images"] / tableau["images"].sum()).round(1)
display(tableau)
ratio_desequilibre = tableau["images"].max() / tableau["images"].min()
print("Rapport classe la plus grande / la plus petite : %.1f" % ratio_desequilibre)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
sns.barplot(x=tableau.index, y=tableau["images"], ax=ax[0], color="steelblue"); ax[0].set_title("Images par émotion")
sns.barplot(x=tableau.index, y=tableau["sujets"], ax=ax[1], color="indianred"); ax[1].set_title("Sujets par émotion")
for a in ax: a.tick_params(axis="x", rotation=40)
plt.tight_layout(); plt.show()

# ----- exemples
fig, axes = plt.subplots(len(CLASSES), 4, figsize=(7, 1.9 * len(CLASSES)))
for ligne, emo in enumerate(CLASSES):
    choix = rng.choice(np.where(df.emotion.values == emo)[0], 4, replace=False)
    for col, i in enumerate(choix):
        axes[ligne, col].imshow(X_img[i], cmap="gray"); axes[ligne, col].axis("off")
        axes[ligne, col].set_title("%s | %s" % (emo, df.subject[i]), fontsize=7)
plt.tight_layout(); plt.show()

### Réponses — Partie 1
1. **Équilibré ?** Non : voir le tableau (rapport classe max / classe min). Les classes *contempt*, *fear* et *sadness* sont les plus petites.
2. **Images ≠ sujets.** Chaque sujet apporte quelques images par émotion (frames successives d'une même séquence). Une classe riche en images peut donc avoir peu de personnes : seul le nombre de **sujets** mesure la vraie diversité.
3. **Émotions proches.** *anger / disgust / sadness / contempt* bougent peu et se ressemblent ; *happy* et *surprise* sont très marquées.
4. **Variations.** Éclairage et cadrage sont très contrôlés en laboratoire : peu de variations, ce qui rend le problème plus facile que dans la vraie vie.
5. **Limite.** Expressions **jouées**, exagérées, de face : un bon score sur CK+ ne garantit pas de bons résultats sur des émotions spontanées (ni devant une caméra).
6. **Copies.** Chaque image existait en double dans l'archive : les copies ont été supprimées pour ne pas compter deux fois la même information.

# Partie 2 — Découpage par sujet et déséquilibre

- On sépare les **personnes** (pas les images) : 80 % des sujets pour l'apprentissage, 20 % pour le test.
- Tous les réglages se font par **validation croisée par sujet** sur l'apprentissage seul. Le test n'est regardé qu'à la fin.
- **Déséquilibre** : on donne plus de poids aux classes rares (`class_weight="balanced"`), calculé uniquement sur l'apprentissage.
- **Métrique principale : macro-F1** (moyenne du F1 de chaque classe, toutes comptent pareil). On affiche aussi l'accuracy.

In [ ]:
n_classes = len(CLASSES)


def decouper_par_sujet(seed=SEED):
    for essai in range(100):
        sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed + essai)
        train, test = next(sgkf.split(np.zeros(len(y)), y, groupes))
        if np.bincount(y[train], minlength=n_classes).min() >= 2 and np.bincount(y[test], minlength=n_classes).min() >= 2:
            return train, test
    raise RuntimeError("découpage impossible")


tr_idx, te_idx = decouper_par_sujet()
y_tr, y_te = y[tr_idx], y[te_idx]
g_tr, g_te = groupes[tr_idx], groupes[te_idx]

# ----- preuve : aucun sujet en commun
communs = set(g_tr) & set(g_te)
assert len(communs) == 0 and set(tr_idx).isdisjoint(te_idx)
print("Apprentissage : %d images, %d sujets | Test : %d images, %d sujets" % (len(tr_idx), len(set(g_tr)), len(te_idx), len(set(g_te))))
print("Sujets communs :", len(communs), "-> aucune fuite")

# ----- plis de validation croisée (par sujet), identiques pour toutes les expériences
cv_splits = list(StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(len(y_tr)), y_tr, g_tr))
for a, b in cv_splits:
    assert set(g_tr[a]).isdisjoint(g_tr[b])
print("Plis de validation : aucun sujet partagé entre apprentissage et validation")

repartition = pd.DataFrame({"train": np.bincount(y_tr, minlength=n_classes), "test": np.bincount(y_te, minlength=n_classes)}, index=CLASSES)
display(repartition)

# ----- déséquilibre : poids calculés sur l'apprentissage seulement
sw_tr = compute_sample_weight("balanced", y_tr)

### Réponses — Partie 2
1. **Pourquoi pas un tirage au hasard d'images ?** Un même sujet fournit plusieurs images quasi identiques : au hasard, il se retrouve dans l'apprentissage **et** le test. Le modèle reconnaît alors la *personne* et non l'émotion → score trop beau.
2. **Vérification.** Les `assert` ci-dessus prouvent qu'aucun sujet n'est commun entre apprentissage et test, ni entre les plis de validation.
3. **Déséquilibre.** Pondération des classes : simple, sans inventer d'images, et sans toucher au test.
4. **Métriques.** Macro-F1 (principale), accuracy équilibrée, accuracy, matrices de confusion.
5. **Pourquoi macro-F1 ?** L'accuracy est dominée par les grosses classes ; la macro-F1 pénalise une petite classe mal reconnue.

## Outils communs (évaluation et réglage des classifieurs)

`tune(...)` fait toujours la même chose, pour tous les descripteurs : chercher les meilleurs réglages par **validation croisée par sujet** (critère macro-F1), puis évaluer **une seule fois** sur le test.

In [ ]:
def metriques(y_vrai, y_pred):
    return {"accuracy": accuracy_score(y_vrai, y_pred),
            "balanced_acc": balanced_accuracy_score(y_vrai, y_pred),
            "macro_f1": f1_score(y_vrai, y_pred, average="macro", zero_division=0)}


class XGBRobuste(BaseEstimator, ClassifierMixin):
    """XGBoost qui accepte des étiquettes quelconques (même si une classe manque dans un pli)."""

    def __init__(self, n_estimators=100, max_depth=3, learning_rate=0.1, subsample=1.0, colsample_bytree=0.3):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree

    def fit(self, X, y, sample_weight=None):
        self.classes_ = np.unique(y)
        y_local = np.searchsorted(self.classes_, y)
        self.modele_ = XGBClassifier(n_estimators=self.n_estimators, max_depth=self.max_depth,
                                     learning_rate=self.learning_rate, subsample=self.subsample,
                                     colsample_bytree=self.colsample_bytree, tree_method="hist", max_bin=64,
                                     n_jobs=1, random_state=SEED, verbosity=0)
        self.modele_.fit(X, y_local, sample_weight=sample_weight)
        return self

    def predict(self, X):
        return self.classes_[self.modele_.predict(X)]


def construire(nom, blocks=None, k=None, probability=False):
    """Pipeline : [mise à l'échelle (SVM)] -> [sélection ANOVA] -> classifieur."""
    etapes = []
    if nom == "SVM":
        etapes.append(("scale", BlockScaler(blocks)))
    if k:
        etapes.append(("select", SelectKBest(f_classif, k=k)))
    if nom == "SVM":
        clf = SVC(class_weight="balanced", probability=probability, random_state=SEED)
    elif nom == "RF":
        clf = RandomForestClassifier(class_weight="balanced_subsample", n_jobs=1, random_state=SEED)
    else:
        clf = XGBRobuste()
    etapes.append(("clf", clf))
    return Pipeline(etapes)


GRILLES = {
    "SVM": [{"clf__kernel": ["rbf"], "clf__C": [1, 10, 100, 1000], "clf__gamma": [0.003, 0.01, 0.03, 0.1, 0.3, 1]},
            {"clf__kernel": ["linear"], "clf__C": [0.1, 1, 10, 100]}],
    "RF": {"clf__n_estimators": [200, 400], "clf__max_depth": [None, 20], "clf__max_features": ["sqrt", 0.1],
           "clf__min_samples_leaf": [1, 2]},
    "XGB": {"clf__n_estimators": [100, 200], "clf__max_depth": [3, 5], "clf__learning_rate": [0.1, 0.2],
            "clf__colsample_bytree": [0.1, 0.3], "clf__subsample": [0.7, 1.0]},
}

RESULTATS, MODELES = [], {}


def tune(representation, X, classifieurs=("SVM", "RF", "XGB"), blocks=None, k=None):
    """Cherche les meilleurs réglages (validation croisée par sujet), puis évalue sur le test."""
    X_tr, X_te = X[tr_idx], X[te_idx]
    for nom in classifieurs:
        modele = construire(nom, blocks, k)
        if nom == "SVM":
            recherche = GridSearchCV(modele, GRILLES["SVM"], scoring="f1_macro", cv=cv_splits, n_jobs=-1)
        else:
            recherche = RandomizedSearchCV(modele, GRILLES[nom], n_iter=N_ITER, scoring="f1_macro",
                                           cv=cv_splits, n_jobs=-1, random_state=SEED)
        options = {"clf__sample_weight": sw_tr} if nom == "XGB" else {}
        debut = time.perf_counter()
        recherche.fit(X_tr, y_tr, **options)
        duree = time.perf_counter() - debut

        i = recherche.best_index_
        plis = [recherche.cv_results_["split%d_test_score" % f][i] for f in range(N_FOLDS)]
        debut = time.perf_counter()
        pred = recherche.predict(X_te)
        ms_par_image = (time.perf_counter() - debut) / len(X_te) * 1000

        ligne = {"representation": representation, "classifieur": nom, "dim": X.shape[1], "dim_utilisee": k or X.shape[1],
                 "params": {c.replace("clf__", ""): v for c, v in recherche.best_params_.items()},
                 "cv_f1": float(np.mean(plis)), "cv_std": float(np.std(plis)),
                 "temps_fit_s": float(recherche.refit_time_), "temps_pred_ms": ms_par_image}
        ligne.update({"test_" + a: b for a, b in metriques(y_te, pred).items()})
        RESULTATS.append(ligne)
        MODELES[(representation, nom)] = {"pred": pred, "blocks": blocks, "k": k, "X": X, "params": ligne["params"]}
        print("[%-26s %-3s] validation macro-F1 = %.3f ± %.3f | test macro-F1 = %.3f | %.0fs"
              % (representation, nom, ligne["cv_f1"], ligne["cv_std"], ligne["test_macro_f1"], duree))


COLONNES = ["representation", "classifieur", "dim_utilisee", "params", "cv_f1", "cv_std", "test_macro_f1",
            "test_accuracy", "temps_fit_s", "temps_pred_ms"]


def resultats(prefixe=None, exact=None):
    d = pd.DataFrame(RESULTATS)
    if prefixe:
        d = d[d.representation.str.startswith(prefixe)]
    if exact:
        d = d[d.representation.isin(exact)]
    return d[COLONNES].round(3)


def meilleur(nom, exact=False):
    """Meilleure ligne (choisie sur la VALIDATION, jamais sur le test)."""
    d = pd.DataFrame(RESULTATS)
    d = d[d.representation == nom] if exact else d[d.representation.str.startswith(nom)]
    return d.sort_values("cv_f1", ascending=False).iloc[0]


def cv_rapide(X, blocks=None):
    """Comparaison rapide de configurations : SVM linéaire, macro-F1 en validation croisée."""
    modele = Pipeline([("scale", BlockScaler(blocks)), ("clf", SVC(kernel="linear", C=10, class_weight="balanced"))])
    scores = cross_val_score(modele, X[tr_idx], y_tr, cv=cv_splits, scoring="f1_macro", n_jobs=-1)
    return scores.mean(), scores.std()


def matrice_confusion(ax, y_vrai, y_pred, titre):
    cm = confusion_matrix(y_vrai, y_pred, labels=range(n_classes), normalize="true")
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, cbar=False, ax=ax,
                xticklabels=CLASSES, yticklabels=CLASSES, annot_kws={"size": 7})
    ax.set_title(titre, fontsize=9); ax.set_xlabel("prédit"); ax.set_ylabel("vrai"); ax.tick_params(axis="x", rotation=45)


def confusions_principales(y_vrai, y_pred, n=4):
    cm = confusion_matrix(y_vrai, y_pred, labels=range(n_classes), normalize="true")
    paires = [(CLASSES[i], CLASSES[j], round(float(cm[i, j]), 2)) for i in range(n_classes) for j in range(n_classes) if i != j]
    return sorted(paires, key=lambda t: -t[2])[:n]

# Partie 3 — HOG

**Principe.** L'image est découpée en petites *cellules* ; dans chacune, on compte la **direction des contours** (gradient). On obtient la forme du visage : sourcils, paupières, plis, contour des lèvres.

In [ ]:
exemples = [c for c in ["happy", "surprise", "anger"] if c in CLASSES]
tailles_cellule = [4, 8, 12]
fig, axes = plt.subplots(len(exemples), 2 + len(tailles_cellule), figsize=(2.3 * (2 + len(tailles_cellule)), 2.3 * len(exemples)))
for ligne, emo in enumerate(exemples):
    img = X_img[np.where(df.emotion.values == emo)[0][0]]
    gx = cv2.Sobel(img.astype(np.float32), cv2.CV_32F, 1, 0)
    gy = cv2.Sobel(img.astype(np.float32), cv2.CV_32F, 0, 1)
    axes[ligne, 0].imshow(img, cmap="gray"); axes[ligne, 0].set_title("original (%s)" % emo, fontsize=8)
    axes[ligne, 1].imshow(np.hypot(gx, gy), cmap="magma"); axes[ligne, 1].set_title("gradient", fontsize=8)
    for col, t in enumerate(tailles_cellule):
        _, vue = hog(img, orientations=9, pixels_per_cell=(t, t), cells_per_block=(2, 2), visualize=True)
        axes[ligne, 2 + col].imshow(vue, cmap="gray"); axes[ligne, 2 + col].set_title("HOG, cellule %dx%d" % (t, t), fontsize=8)
    for a in axes[ligne]:
        a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
HOG_CONFIGS = {
    "cell4":      dict(orientations=9,  pixels_per_cell=(4, 4),   cells_per_block=(2, 2)),
    "cell6":      dict(orientations=9,  pixels_per_cell=(6, 6),   cells_per_block=(2, 2)),
    "cell8":      dict(orientations=9,  pixels_per_cell=(8, 8),   cells_per_block=(2, 2)),
    "cell12":     dict(orientations=9,  pixels_per_cell=(12, 12), cells_per_block=(2, 2)),
    "cell6_or12": dict(orientations=12, pixels_per_cell=(6, 6),   cells_per_block=(2, 2)),
    "cell6_bloc3": dict(orientations=9, pixels_per_cell=(6, 6),   cells_per_block=(3, 3)),
}


def extraire_hog(X, **options):
    return np.array([hog(im, block_norm="L2-Hys", **options) for im in X], dtype=np.float32)


HOG_X, lignes = {}, []
for nom, options in HOG_CONFIGS.items():
    debut = time.perf_counter()
    HOG_X[nom] = extraire_hog(X_img, **options)
    duree = time.perf_counter() - debut
    moyenne, ecart = cv_rapide(HOG_X[nom])
    lignes.append({"config": nom, "dimension": HOG_X[nom].shape[1], "temps_s": round(duree, 2),
                   "macro_f1_cv": round(moyenne, 3), "ecart_type": round(ecart, 3)})
etude_hog = pd.DataFrame(lignes).sort_values("macro_f1_cv", ascending=False)
display(etude_hog)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
sns.barplot(data=etude_hog, x="config", y="macro_f1_cv", ax=ax[0], color="steelblue"); ax[0].set_title("Performance")
sns.barplot(data=etude_hog, x="config", y="dimension", ax=ax[1], color="indianred"); ax[1].set_title("Dimension")
sns.barplot(data=etude_hog, x="config", y="temps_s", ax=ax[2], color="seagreen"); ax[2].set_title("Temps d'extraction (s)")
for a in ax:
    a.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

In [ ]:
# les 2 meilleures configurations (choisies sur la validation) x 3 classifieurs
for nom in etude_hog.config.head(2):
    tune("HOG|" + nom, HOG_X[nom])
display(resultats("HOG|"))

b = meilleur("HOG|")
HOG_CFG = b.representation.split("|")[1]
print("\n=> Meilleur HOG : %s + %s (validation %.3f)" % (HOG_CFG, b.classifieur, b.cv_f1))

pred = MODELES[(b.representation, b.classifieur)]["pred"]
fig, ax = plt.subplots(figsize=(5, 4.3)); matrice_confusion(ax, y_te, pred, "HOG %s + %s" % (HOG_CFG, b.classifieur)); plt.show()
print("Confusions les plus fréquentes :", confusions_principales(y_te, pred))

### Réponses — Partie 3
1. **Ce que HOG montre.** Les contours : sourcils, paupières, plis du front et du nez, bouche.
2. **Taille des cellules.** Petites cellules = détail fin mais plus de bruit ; grandes cellules = plus robuste mais plus flou (voir les images HOG).
3. **Compromis.** Diviser la cellule par 2 multiplie la dimension par ~4 : plus de précision, mais plus de calcul et plus de risque de sur-apprentissage (voir le tableau).
4. **Paramètres influents.** Comparer, dans le tableau, la taille de cellule, puis les orientations et la taille de bloc.
5. **Meilleur classifieur.** Voir le tableau : le SVM est en général le plus à l'aise avec beaucoup de caractéristiques corrélées.
6. **Validation et test cohérents ?** Comparer les colonnes `cv_f1` et `test_macro_f1` ; un petit écart est normal (test = peu de personnes).
7. **Émotions difficiles.** Voir les confusions : en général *anger / sadness / contempt / disgust*.

# Partie 4 — LBP

**Principe.** Pour chaque pixel, on compare sa luminosité à celle de ses voisins : on obtient un code qui décrit la **micro-texture** (bord, coin, zone plate). Un **histogramme** de ces codes résume l'image ; en le calculant **par cellule** (grille), on garde l'information « où ».

In [ ]:
img = X_img[np.where(df.emotion.values == exemples[0])[0][0]]
fig, axes = plt.subplots(1, 5, figsize=(14, 2.8))
axes[0].imshow(img, cmap="gray"); axes[0].set_title("original")
for a, (P, R) in zip(axes[1:3], [(8, 1), (16, 2)]):
    a.imshow(local_binary_pattern(img, P, R, "uniform"), cmap="nipy_spectral"); a.set_title("LBP P=%d, R=%d" % (P, R))
axes[3].bar(range(10), lbp_histogram(img, 8, 1, (1, 1))); axes[3].set_title("histogramme global")
axes[4].imshow(lbp_histogram(img, 8, 1, (6, 6)).reshape(36, 10), cmap="viridis", aspect="auto"); axes[4].set_title("histogrammes par cellule (6x6)")
for a in axes[:3]:
    a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
LBP_CONFIGS = {
    "P8R1_g1":  dict(P=8,  R=1, grid=(1, 1)),
    "P8R1_g4":  dict(P=8,  R=1, grid=(4, 4)),
    "P8R1_g6":  dict(P=8,  R=1, grid=(6, 6)),
    "P8R1_g8":  dict(P=8,  R=1, grid=(8, 8)),
    "P8R2_g6":  dict(P=8,  R=2, grid=(6, 6)),
    "P16R2_g6": dict(P=16, R=2, grid=(6, 6)),
}

LBP_X, lignes = {}, []
for nom, options in LBP_CONFIGS.items():
    debut = time.perf_counter()
    LBP_X[nom] = np.array([lbp_histogram(im, **options) for im in X_img], dtype=np.float32)
    duree = time.perf_counter() - debut
    moyenne, ecart = cv_rapide(LBP_X[nom])
    lignes.append({"config": nom, "dimension": LBP_X[nom].shape[1], "temps_s": round(duree, 2),
                   "macro_f1_cv": round(moyenne, 3), "ecart_type": round(ecart, 3)})
etude_lbp = pd.DataFrame(lignes).sort_values("macro_f1_cv", ascending=False)
display(etude_lbp)

for nom in etude_lbp.config.head(2):
    tune("LBP|" + nom, LBP_X[nom])
display(resultats("LBP|"))

b = meilleur("LBP|")
LBP_CFG = b.representation.split("|")[1]
print("\n=> Meilleur LBP : %s + %s (validation %.3f)" % (LBP_CFG, b.classifieur, b.cv_f1))

bh = meilleur("HOG|")
pred_h, pred_l = MODELES[(bh.representation, bh.classifieur)]["pred"], MODELES[(b.representation, b.classifieur)]["pred"]
fig, ax = plt.subplots(1, 2, figsize=(11, 4.3))
matrice_confusion(ax[0], y_te, pred_h, "HOG (meilleur)"); matrice_confusion(ax[1], y_te, pred_l, "LBP (meilleur)")
plt.tight_layout(); plt.show()
print("Confusions HOG :", confusions_principales(y_te, pred_h))
print("Confusions LBP :", confusions_principales(y_te, pred_l))
print("HOG et LBP donnent la même prédiction sur %.0f %% des images de test" % (100 * np.mean(pred_h == pred_l)))

### Réponses — Partie 4
1. **Information décrite.** La micro-texture locale : bords, coins, plis, rugosité.
2. **Rayon et voisins.** Petit rayon = détails fins ; grand rayon = structures plus larges mais plus de dimensions (voir `etude_lbp`).
3. **Perte avec un histogramme global.** On perd *où* sont les motifs (bouche ou sourcils ?). La grille d'histogrammes par cellule le restaure.
4. **Meilleur classifieur.** Voir le tableau.
5. **LBP vs HOG.** LBP ignore les changements globaux de luminosité mais est sensible au bruit et au décalage ; HOG normalise localement le contraste.
6. **Mêmes confusions ?** Comparer les confusions et le taux d'accord ci-dessus : un accord partiel = complémentarité.

# Partie 5 — Fusion HOG + LBP

On **colle** les deux vecteurs (fusion précoce). Comme HOG a beaucoup plus de colonnes que LBP, `BlockScaler` donne à chaque famille **le même poids total** (sinon HOG écraserait LBP).

In [ ]:
X_hog, X_lbp = HOG_X[HOG_CFG], LBP_X[LBP_CFG]
d_hog, d_lbp = X_hog.shape[1], X_lbp.shape[1]
X_hl = np.hstack([X_hog, X_lbp])
BLOCS_HL = [(0, d_hog), (d_hog, d_hog + d_lbp)]
print("HOG %s (%d) + LBP %s (%d) = %d dimensions" % (HOG_CFG, d_hog, LBP_CFG, d_lbp, X_hl.shape[1]))

tune("HOG+LBP", X_hl, blocks=BLOCS_HL)
bh, bl = meilleur("HOG|"), meilleur("LBP|")
comparaison = resultats(exact=[bh.representation, bl.representation, "HOG+LBP"]).sort_values(["classifieur", "representation"])
display(comparaison)

# F1 par classe : la fusion aide-t-elle toutes les classes ?
f1_classe = {}
for nom, (rep, clf) in {"HOG": (bh.representation, bh.classifieur), "LBP": (bl.representation, bl.classifieur),
                        "HOG+LBP": ("HOG+LBP", meilleur("HOG+LBP", exact=True).classifieur)}.items():
    f1_classe[nom] = f1_score(y_te, MODELES[(rep, clf)]["pred"], average=None, labels=range(n_classes), zero_division=0)
display(pd.DataFrame(f1_classe, index=CLASSES).round(2))

### Réponses — Partie 5
1. **Complémentaires ?** HOG = forme des contours, LBP = texture fine : deux points de vue différents sur le même visage.
2. **Toutes les métriques ?** Comparer `cv_f1`, `test_macro_f1`, `test_accuracy` : une amélioration en validation n'est pas toujours confirmée sur le test.
3. **Toutes les classes ?** Voir le tableau F1 par classe : la fusion aide certaines classes, pas forcément toutes.
4. **Dimension et temps.** Dimensions et temps s'additionnent (colonnes `dim_utilisee`, `temps_fit_s`, `temps_pred_ms`).
5. **Plus grand = meilleur ?** Non : avec peu de personnes, trop de colonnes redondantes font sur-apprendre. D'où la Partie 6.

# Partie 6 — Sélection de caractéristiques

**Méthode : filtre ANOVA** (`SelectKBest`). Pour chaque colonne, on mesure si sa valeur **change vraiment d'une émotion à l'autre** par rapport à sa variation normale (statistique F). On garde les `k` meilleures.

- **Pourquoi ?** Rapide, ne dépend pas du classifieur, adapté à des milliers de colonnes.
- **Choix de `k`.** Courbe macro-F1(k) en validation croisée par sujet ; on prend le **plus petit `k` à moins d'une erreur-type du meilleur**.
- **Sans fuite.** La sélection est *dans* le pipeline : elle est réapprise à chaque pli, sur les données d'apprentissage uniquement.

In [ ]:
def choisir_k(X, blocks=None):
    ks = [k for k in [25, 50, 100, 200, 400, 800, 1600, 3200] if k < X.shape[1]] + [X.shape[1]]
    lignes = []
    for k in ks:
        modele = Pipeline([("scale", BlockScaler(blocks)), ("select", SelectKBest(f_classif, k=k)),
                           ("clf", SVC(kernel="linear", C=1, class_weight="balanced"))])
        scores = cross_val_score(modele, X[tr_idx], y_tr, cv=cv_splits, scoring="f1_macro", n_jobs=-1)
        lignes.append({"k": k, "moyenne": scores.mean(), "erreur_type": scores.std() / np.sqrt(len(scores))})
    courbe = pd.DataFrame(lignes)
    i = courbe.moyenne.idxmax()
    seuil = courbe.moyenne[i] - courbe.erreur_type[i]
    return int(courbe[courbe.moyenne >= seuil].k.min()), courbe


def tracer_courbe_k(courbe, k_choisi, titre):
    plt.figure(figsize=(6, 3.3))
    plt.errorbar(courbe.k, courbe.moyenne, yerr=courbe.erreur_type, marker="o", capsize=3)
    plt.axvline(k_choisi, color="r", ls="--", label="k choisi = %d" % k_choisi)
    plt.xscale("log"); plt.xlabel("k (nombre de caractéristiques)"); plt.ylabel("macro-F1 (validation)")
    plt.title(titre); plt.legend(); plt.show()


K_HL, courbe_hl = choisir_k(X_hl, BLOCS_HL)
tracer_courbe_k(courbe_hl, K_HL, "Choix de k (HOG+LBP)")
tune("SELECT|HOG+LBP|k%d" % K_HL, X_hl, blocks=BLOCS_HL, k=K_HL)
display(resultats(exact=["HOG+LBP", "SELECT|HOG+LBP|k%d" % K_HL]).sort_values(["classifieur", "representation"]))

### Réponses — Partie 6
1. **Pourquoi l'ANOVA ?** Voir l'encadré : rapide, indépendante du classifieur, adaptée à beaucoup de colonnes.
2. **Choix de k.** Par la courbe ci-dessus (règle de l'erreur-type), en validation croisée sur l'apprentissage.
3. **Quand la sélection est-elle apprise ?** Dans chaque pli d'apprentissage (et une dernière fois sur tout l'apprentissage), jamais avec la validation ni le test.
4. **Généralisation ou vitesse ?** Comparer fusion complète et sélection : mêmes scores avec moins de colonnes = gain de vitesse ; meilleurs scores = effet de régularisation.
5. **Risque si on sélectionne avant la validation.** La sélection « voit » les étiquettes de la validation et du test : elle garde les colonnes qui séparent *ces* images, et les scores sont gonflés.
6. **Le meilleur classifieur change-t-il ?** Comparer le classement avant/après dans le tableau.

# Partie 7 — Bilan du TP1 (sans landmarks)

In [ ]:
tp1 = pd.DataFrame(RESULTATS)
tp1 = tp1[tp1.representation.str.startswith(("HOG|", "LBP|")) | (tp1.representation == "HOG+LBP") | tp1.representation.str.startswith("SELECT|HOG+LBP")]
display(tp1.sort_values("cv_f1", ascending=False)[COLONNES].round(3).reset_index(drop=True))

# Partie 8 — Suite du TP : landmarks du visage

## 8.1 État de l'art (en bref)

| Travail | Idée | Ce qu'on en retient |
|---|---|---|
| **Lucey et al., 2010** (CK+) | CK+ est livré avec **68 landmarks** par image et des séquences allant du visage neutre au pic d'expression. | Les 68 points sont le standard pour CK+. |
| **Kazemi & Sullivan, 2014** (détecteur de **dlib**) | Cascade d'arbres de régression : détecte les 68 points en quelques millisecondes sur un simple processeur. | **Notre détecteur** : assez rapide pour une caméra en direct. |
| **Kumar et al., 2020** (**LUVLi**, CVPR) | Réseau profond qui prédit pour chaque point sa **position, son incertitude et sa visibilité** ; l'incertitude sert à repérer les alignements ratés. | Plus précis mais lourd (licence MERL, ≈ 565 Mo de poids, GPU). Non retenu pour le direct (voir ci-dessous). |
| **arXiv:1701.01879** | Distances entre **toutes les paires** des 68 points (2278 paires) + SVM, sur CK+. | Les distances sont une base solide ; on en garde un jeu **compact**. |
| **Suivi de landmarks + triangles + AdaBoost** (IJMUE) | Aires/formes de **triangles** formés par les landmarks, sélectionnées par AdaBoost, puis SVM (97,8 % rapportés sur CK+). | Aires de triangles + **sélection** de caractéristiques. |
| **Liu, Cheng & Lee, 2020** (IEEE Sensors J.) | **Courbure** des landmarks + « landmarks vectorisés », SVM réglé par algorithme génétique (95,9 % / 97,6 % de précision test rapportés sur CK+ à 8 / 7 classes). | **Courbure** des sourcils, des lèvres, de la mâchoire. |
| **Happy & Routray, 2015** (arXiv:1505.04030) | Les landmarks servent à localiser des **zones actives** du visage ; **PHOG + LBP** y sont calculés, puis SVM. | **HOG + LBP calculés sur des patchs** yeux / nez / bouche. |
| **Liu et al., 2017** (Sensors 17(4):712) | **LBP + HOG** sur des zones saillantes du visage, puis réduction par PCA. | Justifie la fusion HOG+LBP et la mise à l'échelle des blocs. |
| **GLaRE (arXiv:2508.20579)**, enquête de Gueuret et al. (CBMI 2024) | Les landmarks forment un **graphe** exploité par des réseaux de neurones sur graphes. | Piste d'amélioration (hors périmètre du TP). |

*Les scores publiés ne sont pas comparables aux nôtres : protocoles souvent différents, alors que nous séparons strictement les personnes et prédisons image par image.*

## 8.2 Idée retenue

Combiner trois idées de la littérature, avec des outils **assez légers pour une caméra** :
1. **Mesures géométriques compactes** (distances, angles, courbures, aires de triangles) calculées sur les 68 points — *Liu 2020, IJMUE, arXiv:1701.01879*.
2. **Patchs HOG + LBP** découpés **grâce aux landmarks** (yeux + sourcils, nez, bouche) — *Happy & Routray 2015*.
3. **Fusion** de tout cela avec HOG et LBP du TP1, puis **sélection** de caractéristiques — *Liu 2017*.

**Pourquoi dlib ?** Le sujet autorise dlib ou LUVLi. On prend dlib : ~8 ms par visage sur CPU, sans GPU, installable en une ligne, donc utilisable en direct sur une caméra. LUVLi (Kumar 2020) est cité dans l'état de l'art mais n'est pas utilisé : son code officiel date de Python 2.7 / PyTorch 0.3 et ne se met pas en place simplement sur Kaggle.

*Note : une première version de ce TP essayait d'utiliser l'**incertitude** des landmarks (idée de LUVLi) comme caractéristique, estimée en répétant la détection sur des images légèrement décalées. Le gain restait dans le bruit de la validation et les images mal classées n'étaient pas plus incertaines que les autres : l'idée a été abandonnée.*

## 8.3 Détection des 68 landmarks et mesures

Pour chaque visage 48×48 : `face_features(...)` (dans `emotion_landmarks.py`) fait
1. **68 landmarks** avec dlib (moyenne de quelques vues légèrement décalées pour plus de stabilité) ;
2. **alignement** : on met les yeux à l'horizontale et on prend la **distance entre les yeux** comme unité → les mesures ne dépendent ni de la taille du visage, ni de sa position, ni de sa rotation dans l'image ;
3. **80 mesures géométriques** : distances (sourcil–œil, largeur et ouverture de la bouche, nez–lèvre, coins de la bouche…), ouverture des yeux et de la bouche, hauteur des commissures (sourire), angles, courbures, aires de triangles ;
4. **984 caractéristiques de patchs** : HOG + LBP sur les zones yeux/sourcils, nez, bouche.

In [ ]:
detecteur, predicteur = load_dlib()

empreinte = hashlib.md5(X_img.tobytes()).hexdigest()[:8]
cache = Path("features_landmarks_%s_v%d.npz" % (empreinte, N_VIEWS))
if cache.exists():
    donnees = np.load(cache, allow_pickle=True)
    X_lm, LANDMARKS, NOMS_GEO = donnees["X"], donnees["landmarks"], list(donnees["noms"])
    print("Chargé depuis", cache)
else:
    vecteurs, points = [], []
    debut = time.perf_counter()
    for i, img in enumerate(X_img):
        vecteur, lm, mesures = face_features(img, predicteur, N_VIEWS)
        vecteurs.append(vecteur)
        points.append(lm)
        if (i + 1) % 200 == 0:
            print("  %d/%d images (%.0fs)" % (i + 1, len(X_img), time.perf_counter() - debut))
    X_lm, LANDMARKS, NOMS_GEO = np.array(vecteurs), np.array(points), list(mesures.keys())
    np.savez(cache, X=X_lm, landmarks=LANDMARKS, noms=np.array(NOMS_GEO))

N_GEO = len(NOMS_GEO)
X_geo, X_patch = X_lm[:, :N_GEO], X_lm[:, N_GEO:]
print("Mesures géométriques : %d | caractéristiques de patchs : %d | total : %d" % (N_GEO, X_patch.shape[1], X_lm.shape[1]))

# ----- on regarde si les points sont bien placés (1 visage par émotion)
fig, axes = plt.subplots(1, n_classes, figsize=(2 * n_classes, 2.3))
for a, emo in zip(axes, CLASSES):
    i = np.where(df.emotion.values == emo)[0][0]
    a.imshow(X_img[i], cmap="gray"); a.scatter(LANDMARKS[i, :, 0], LANDMARKS[i, :, 1], s=6, c="lime")
    a.set_title(emo, fontsize=8); a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ----- les mesures « parlent » : moyenne par émotion (en unités de distance entre les yeux)
colonnes = [NOMS_GEO.index(n) for n in MESURES_AFFICHEES]
moyennes = pd.DataFrame(X_geo[:, colonnes], columns=MESURES_AFFICHEES).groupby(df.emotion.values).mean().loc[CLASSES]
display(moyennes.round(2).style.background_gradient(axis=0, cmap="Blues"))

# ----- cadrage : comment dlib « voit » les visages CK+ (sert à découper les visages de la caméra de la même façon)
FRAMING, taux_detection = measure_framing(X_img[rng.permutation(len(X_img))], detecteur)
print("dlib détecte le visage dans %.0f %% des images CK+ ; le visage détecté occupe %.0f %% de la largeur (décalage vertical %.2f)"
      % (100 * taux_detection, 100 * FRAMING[0], FRAMING[1]))

## 8.4 Expériences

On compare, avec le même protocole que le TP1 :
- les **mesures géométriques seules** (`GEO`), les **patchs seuls** (`PATCHS`), et les deux ensemble (`GEO+PATCHS`) ;
- la **fusion avec HOG+LBP** du TP1 ;
- la **sélection** de caractéristiques sur la meilleure combinaison.

In [ ]:
BLOCS_GP = [(0, N_GEO), (N_GEO, X_lm.shape[1])]

tune("GEO", X_geo)                                          # 3 classifieurs
tune("PATCHS", X_patch, ("SVM",))
tune("GEO+PATCHS", X_lm, blocks=BLOCS_GP)                   # 3 classifieurs

# ----- fusion avec les descripteurs du TP1
d_hl = X_hl.shape[1]
X_f1 = np.hstack([X_hl, X_geo])
tune("HOG+LBP+GEO", X_f1, ("SVM",), blocks=BLOCS_HL + [(d_hl, d_hl + N_GEO)])
X_f2 = np.hstack([X_hl, X_lm])
tune("HOG+LBP+GEO+PATCHS", X_f2, ("SVM",), blocks=BLOCS_HL + [(d_hl, d_hl + N_GEO), (d_hl + N_GEO, X_f2.shape[1])])

# ----- sélection de caractéristiques sur GEO+PATCHS
K_GP, courbe_gp = choisir_k(X_lm, BLOCS_GP)
tracer_courbe_k(courbe_gp, K_GP, "Choix de k (GEO+PATCHS)")
NOM_SELECT = "SELECT|GEO+PATCHS|k%d" % K_GP
tune(NOM_SELECT, X_lm, blocks=BLOCS_GP, k=K_GP)

display(resultats(exact=["GEO", "PATCHS", "GEO+PATCHS", "HOG+LBP+GEO", "HOG+LBP+GEO+PATCHS", NOM_SELECT]).sort_values("cv_f1", ascending=False))

### Réponses — Partie 8
1. **Méthode de détection.** dlib (68 points, cascade d'arbres de régression), appliqué sur le visage 48×48 agrandi, moyenné sur quelques vues.
2. **Caractéristiques et justification.** Distances, angles, courbures et triangles : chaque mouvement musculaire de l'expression (sourcils levés ou froncés, bouche ouverte, coins relevés, nez plissé) déplace ces points. Normalisées par la distance entre les yeux, elles sont insensibles à la taille, à la position et à la rotation.
3. **Fusion.** Collage des vecteurs avec `BlockScaler` (chaque famille pèse pareil), puis sélection ANOVA en option.
4. **Comparer** `GEO` vs `PATCHS` vs `GEO+PATCHS`, puis avec / sans HOG+LBP : ce que les landmarks apportent, et si HOG+LBP global apporte encore quelque chose.

# Partie 9 — Comparaison finale

Les choix (configurations, classifieur, `k`) sont faits sur la **validation**. Le test est seulement rapporté.

In [ ]:
TOUT = pd.DataFrame(RESULTATS)
display(TOUT.sort_values("cv_f1", ascending=False)[COLONNES].round(3).reset_index(drop=True))
TOUT.drop(columns=["params"]).to_csv("resultats.csv", index=False)

# ----- meilleur modèle de chaque représentation : validation et test
meilleurs = TOUT.loc[TOUT.groupby("representation").cv_f1.idxmax()].sort_values("cv_f1")
pos = np.arange(len(meilleurs))
plt.figure(figsize=(8, 0.45 * len(meilleurs) + 1.5))
plt.barh(pos - 0.2, meilleurs.cv_f1, 0.4, label="validation (CV)")
plt.barh(pos + 0.2, meilleurs.test_macro_f1, 0.4, label="test")
plt.yticks(pos, ["%s (%s)" % (r, c) for r, c in zip(meilleurs.representation, meilleurs.classifieur)], fontsize=8)
plt.xlabel("macro-F1"); plt.legend(); plt.xlim(0, 1); plt.tight_layout(); plt.show()

In [ ]:
# ----- matrices de confusion des 2 meilleurs modèles (classés sur la validation)
top2 = TOUT.sort_values("cv_f1", ascending=False).head(2)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for ax, (_, r) in zip(axes, top2.iterrows()):
    matrice_confusion(ax, y_te, MODELES[(r.representation, r.classifieur)]["pred"], "%s + %s" % (r.representation, r.classifieur))
plt.tight_layout(); plt.show()

# ----- F1 par classe pour les principales représentations (meilleur classifieur de chacune)
noms_reps = [meilleur("HOG|").representation, meilleur("LBP|").representation, "HOG+LBP", "GEO", "GEO+PATCHS"]
f1_par_classe = {}
for rep in noms_reps:
    ligne = TOUT[TOUT.representation == rep].sort_values("cv_f1", ascending=False).iloc[0]
    f1_par_classe[rep] = f1_score(y_te, MODELES[(rep, ligne.classifieur)]["pred"], average=None, labels=range(n_classes), zero_division=0)
f1_par_classe = pd.DataFrame(f1_par_classe, index=CLASSES)
f1_par_classe["n_test"] = np.bincount(y_te, minlength=n_classes)
display(f1_par_classe.round(2))

In [ ]:
# ----- exemples bien / mal classés (meilleur modèle selon la validation)
r = TOUT.sort_values("cv_f1", ascending=False).iloc[0]
pred = MODELES[(r.representation, r.classifieur)]["pred"]
bons, faux = np.where(pred == y_te)[0], np.where(pred != y_te)[0]
print("%s + %s : %d erreurs sur %d images de test" % (r.representation, r.classifieur, len(faux), len(y_te)))

fig, axes = plt.subplots(2, 8, figsize=(15, 4.4))
for ligne, indices, titre, couleur in [(0, bons, "correct", "green"), (1, faux, "ERREUR", "red")]:
    choisis = rng.choice(indices, min(8, len(indices)), replace=False) if len(indices) else []
    for col in range(8):
        axes[ligne, col].axis("off")
        if col < len(choisis):
            i = choisis[col]
            axes[ligne, col].imshow(X_img[te_idx[i]], cmap="gray")
            axes[ligne, col].set_title("%s\nvrai: %s\nprédit: %s" % (titre, CLASSES[y_te[i]], CLASSES[pred[i]]), fontsize=7, color=couleur)
plt.tight_layout(); plt.show()

In [ ]:
# ----- effet du traitement du déséquilibre (pondération des classes) sur le meilleur SVM
bs = TOUT[TOUT.classifieur == "SVM"].sort_values("cv_f1", ascending=False).iloc[0]
mod = MODELES[(bs.representation, "SVM")]


def svm_test(class_weight):
    modele = construire("SVM", mod["blocks"], mod["k"])
    modele.set_params(clf__class_weight=class_weight, **{"clf__" + c: v for c, v in mod["params"].items()})
    return modele.fit(mod["X"][tr_idx], y_tr).predict(mod["X"][te_idx])


sans, avec = svm_test(None), svm_test("balanced")
effet = pd.DataFrame({"n_train": np.bincount(y_tr, minlength=n_classes),
                      "F1 sans poids": f1_score(y_te, sans, average=None, labels=range(n_classes), zero_division=0),
                      "F1 avec poids": f1_score(y_te, avec, average=None, labels=range(n_classes), zero_division=0)}, index=CLASSES)
effet["écart"] = effet["F1 avec poids"] - effet["F1 sans poids"]
display(effet.sort_values("n_train").round(3))
print("%s | macro-F1 sans poids = %.3f, avec poids = %.3f" % (bs.representation, f1_score(y_te, sans, average="macro"), f1_score(y_te, avec, average="macro")))

### Réponses — Partie 9 (à compléter avec VOS chiffres du tableau)
1. **Meilleure macro-F1 ?** Regarder la première ligne du tableau (classement sur la validation).
2. **Meilleur compromis performance / dimension / temps ?** Comparer `cv_f1`, `dim_utilisee`, `temps_pred_ms` : un modèle avec peu de colonnes et un score proche du meilleur est préférable pour la caméra.
3. **Classes faciles / difficiles ?** Voir le tableau F1 par classe (attention : `n_test` est petit pour certaines classes).
4. **Confusions récurrentes ?** Voir les matrices de confusion.
5. **Déséquilibre.** Voir l'effet de la pondération sur les petites classes (tableau juste au-dessus).
6. **La fusion est-elle utile ?** Comparer `HOG+LBP` avec `GEO`, `GEO+PATCHS` et `HOG+LBP+GEO+PATCHS`. Tenir compte de l'écart-type `cv_std` : une différence plus petite que lui n'est pas démontrée.
7. **Limites.** Expressions jouées, de face, éclairage de laboratoire, peu de sujets (le test porte sur quelques personnes seulement) ; CK+ n'a pas de classe « neutre ».

# Partie 10 — Modèle final et caméra en direct

**Choix du modèle pour la caméra.** On ne garde que ce qui se calcule vite à partir d'un visage : `GEO`, `GEO+PATCHS` ou `GEO+PATCHS` avec sélection (pas besoin de HOG/LBP sur toute l'image). Parmi eux, on prend le meilleur **en validation**.

1. **Estimation honnête** : on entraîne sans les sujets de test, et on mesure sur eux.
2. **Modèle livré** : on ré-entraîne **avec tous les sujets** (plus de personnes = meilleure généralisation) avec les mêmes réglages, et on sauvegarde `emotion_model.joblib`.

In [ ]:
import joblib

REPS_CAMERA = ["GEO", "GEO+PATCHS", NOM_SELECT]
candidats = pd.DataFrame(RESULTATS)
candidats = candidats[(candidats.classifieur == "SVM") & candidats.representation.isin(REPS_CAMERA)]
choix = candidats.sort_values("cv_f1", ascending=False).iloc[0]
rep = choix.representation
GEO_SEUL = (rep == "GEO")
X_cam = X_geo if GEO_SEUL else X_lm
BLOCS_CAM = None if GEO_SEUL else BLOCS_GP
K_CAM = K_GP if rep.startswith("SELECT") else None
print("Modèle retenu pour la caméra :", rep, "| SVM", choix.params, "| validation macro-F1 = %.3f" % choix.cv_f1)


def pipeline_camera():
    p = construire("SVM", BLOCS_CAM, K_CAM, probability=True)       # probability=True : on veut un % de confiance
    return p.set_params(**{"clf__" + c: v for c, v in choix.params.items()})


# ----- 1) estimation honnête : entraîné SANS les sujets de test
p_eval = pipeline_camera().fit(X_cam[tr_idx], y_tr)
proba = p_eval.predict_proba(X_cam[te_idx])
pred_cam = p_eval.classes_[proba.argmax(axis=1)]
m = metriques(y_te, pred_cam)
sur = proba.max(axis=1) >= 0.5
print("Sur les sujets de test : macro-F1 = %.3f | accuracy = %.3f" % (m["macro_f1"], m["accuracy"]))
print("Confiance >= 50 %% pour %.0f %% des images ; sur celles-là, accuracy = %.3f" % (100 * sur.mean(), accuracy_score(y_te[sur], pred_cam[sur])))
fig, ax = plt.subplots(figsize=(5, 4.3)); matrice_confusion(ax, y_te, pred_cam, "Modèle caméra (sujets de test)"); plt.show()

# ----- 2) modèle livré : entraîné avec TOUS les sujets, puis sauvegardé
modele_final = pipeline_camera().fit(X_cam, y)
joblib.dump({"pipeline": modele_final, "classes": CLASSES, "framing": FRAMING, "n_views": N_VIEWS,
             "geo_only": GEO_SEUL, "representation": rep}, "emotion_model.joblib")
print("Sauvegardé : emotion_model.joblib (%.1f Mo)" % (Path("emotion_model.joblib").stat().st_size / 1e6))

### Test du modèle sauvegardé

`EmotionAnalyzer` (dans `emotion_landmarks.py`) est exactement ce que la caméra utilisera : image → visage → 68 landmarks → mesures → émotion. On vérifie d'abord qu'il redonne les mêmes prédictions que le modèle du notebook, puis on le teste sur une **photo complète**.

In [ ]:
analyseur = EmotionAnalyzer("emotion_model.joblib", min_confidence=0.5)

# ----- cohérence : mêmes prédictions que dans le notebook ?
echantillon = rng.choice(len(X_img), 40, replace=False)
debut = time.perf_counter()
reponses = [analyseur.predict_crop(X_img[i])["emotion"] for i in echantillon]
ms = (time.perf_counter() - debut) / len(echantillon) * 1000
attendu = [CLASSES[modele_final.predict_proba(X_cam[i][None]).argmax()] for i in echantillon]
print("Accord avec le modèle du notebook : %.0f %% | %.1f ms par visage (hors détection du visage)" % (100 * np.mean(np.array(reponses) == np.array(attendu)), ms))

In [ ]:
# ----- test sur une photo complète (ici une photo d'exemple ; remplacez par cv2.imread("ma_photo.jpg"))
from skimage import data
photo = cv2.cvtColor(data.astronaut(), cv2.COLOR_RGB2BGR)

res = analyseur.analyze(photo)
if res is None:
    print("Aucun visage détecté")
else:
    dessin = photo.copy()
    x, y0, w, h = res["box"]
    cv2.rectangle(dessin, (x, y0), (x + w, y0 + h), (255, 120, 0), 2)
    for px, py in res["landmarks"]:
        cv2.circle(dessin, (int(px), int(py)), 2, (0, 255, 0), -1)
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
    ax[0].imshow(cv2.cvtColor(dessin[max(y0 - 60, 0):y0 + h + 60, max(x - 60, 0):x + w + 60], cv2.COLOR_BGR2RGB)); ax[0].axis("off")
    ax[0].set_title("%s (%.0f %%)" % (res["label"], 100 * res["confidence"]))
    ax[1].barh(list(res["probabilities"].keys()), list(res["probabilities"].values()), color="steelblue"); ax[1].set_title("probabilités")
    ax[1].set_xlim(0, 1)
    ax[2].axis("off")
    lignes_mesures = [(n.replace("d_", ""), res["distances"][n], res["distances"][n] * res["iod_px"] if n.startswith("d_") else None)
                      for n in MESURES_AFFICHEES]
    texte = "Mesures (unité = distance entre les yeux = %.0f px)\n\n" % res["iod_px"]
    for nom, valeur, pixels in lignes_mesures:
        texte += "%-26s %.2f%s\n" % (nom, valeur, "  (= %.0f px)" % pixels if pixels is not None else "")
    ax[2].text(0, 1, texte, va="top", family="monospace", fontsize=9)
    plt.tight_layout(); plt.show()
    print("Les modèles CK+ n'ont pas de classe « neutre » : une photo neutre donne souvent une confiance faible (« incertain »).")

### Script de la caméra : `live_camera.py`

La cellule suivante écrit le script. Il détecte le visage, trace les 68 points, **affiche les mesures** et **l'émotion** (probabilités moyennées sur 5 images pour éviter le clignotement ; « incertain » si la confiance est trop basse).

In [ ]:
%%writefile live_camera.py
"""
live_camera.py : emotions + mesures du visage en direct avec la webcam.

Fichiers necessaires dans le meme dossier :
    emotion_landmarks.py   emotion_model.joblib   (+ shape_predictor_68_face_landmarks.dat, telecharge seul)
Installation :  pip install dlib-bin opencv-python scikit-image scikit-learn joblib numpy
Lancement    :  python live_camera.py            (webcam 0)
                python live_camera.py 1          (autre camera)
                python live_camera.py video.mp4  (fichier video)
Touche q pour quitter.
"""
import sys
from collections import deque

import cv2
import numpy as np

from emotion_landmarks import EmotionAnalyzer, MESURES_AFFICHEES

SEUIL_CONFIANCE = 0.50    # en dessous, on affiche "incertain" (CK+ n'a pas de classe "neutre")
LISSAGE = 5               # on moyenne les probabilites des 5 dernieres images


def dessiner(frame, res, label, confiance):
    x, y, w, h = res["box"]
    cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 120, 0), 2)
    for px, py in res["landmarks"]:
        cv2.circle(frame, (int(px), int(py)), 1, (0, 255, 0), -1)
    cv2.putText(frame, "%s (%.0f%%)" % (label, 100 * confiance), (x, max(y - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    ligne = 20
    for nom in MESURES_AFFICHEES:
        valeur = res["distances"][nom]
        texte = "%s: %.2f" % (nom.replace("d_", ""), valeur)
        cv2.putText(frame, texte, (10, ligne), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        ligne += 18
    cv2.putText(frame, "distance entre les yeux: %.0f px" % res["iod_px"], (10, ligne),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)


def main():
    source = sys.argv[1] if len(sys.argv) > 1 else "0"
    camera = cv2.VideoCapture(int(source) if source.isdigit() else source)
    analyseur = EmotionAnalyzer("emotion_model.joblib", min_confidence=SEUIL_CONFIANCE)
    historique = deque(maxlen=LISSAGE)

    while True:
        ok, frame = camera.read()
        if not ok:
            break
        res = analyseur.analyze(frame)
        if res is None:
            historique.clear()
        else:
            historique.append([res["probabilities"][c] for c in analyseur.classes])
            moyenne = np.mean(historique, axis=0)
            meilleur = int(np.argmax(moyenne))
            label = analyseur.classes[meilleur] if moyenne[meilleur] >= SEUIL_CONFIANCE else "incertain"
            dessiner(frame, res, label, float(moyenne[meilleur]))
        cv2.imshow("Emotions (q pour quitter)", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    camera.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

### Utiliser la caméra (sur votre ordinateur)

La dernière cellule du notebook crée **un seul fichier : `emotion_camera_bundle.zip`**. Sur Kaggle : *Save Version → Save & Run All (Commit)*, puis onglet **Output** de la version → télécharger le zip (ou `/kaggle/working/` dans le panneau de droite). Décompressez-le dans un dossier. Il contient :

- `emotion_landmarks.py`
- `emotion_model.joblib`
- `live_camera.py`
- `shape_predictor_68_face_landmarks.dat`
- `resultats.csv` (tableau des résultats)

Puis dans un terminal :

```
pip install dlib-bin opencv-python scikit-image scikit-learn joblib numpy
python live_camera.py
```

(`q` pour quitter ; `python live_camera.py 1` pour une autre caméra ; le fichier dlib du zip est utilisé tel quel ; s'il manque, il se télécharge seul.)

**À savoir.**
- Utilisez la **même version de scikit-learn** que dans ce notebook (sinon le modèle peut ne pas se charger).
- CK+ contient des expressions **jouées**, de face, bien éclairées : devant une caméra, jouez l'expression nettement. Comme il n'existe pas de classe « neutre », un visage détendu donne souvent « incertain ». Le seuil est la variable `SEUIL_CONFIANCE` du script.
- Si les points verts sont mal placés sur le visage, le découpage du visage est à ajuster (`framing` dans le modèle) ; la visualisation du script permet de le voir tout de suite.

In [ ]:
# ----- tout rassembler dans UN zip (sur Kaggle : /kaggle/working/ -> onglet Output -> télécharger)
import zipfile
FICHIERS = ["emotion_model.joblib", "emotion_landmarks.py", "live_camera.py", "shape_predictor_68_face_landmarks.dat",
            "resultats.csv"]
if not Path("shape_predictor_68_face_landmarks.dat").exists():
    load_dlib()                                   # télécharge le modèle de landmarks (internet activé)
with zipfile.ZipFile("emotion_camera_bundle.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in FICHIERS:
        if Path(f).exists():
            z.write(f)
            print("  + %s (%.1f Mo)" % (f, Path(f).stat().st_size / 1e6))
        else:
            print("  - absent :", f)
print("-> emotion_camera_bundle.zip (%.0f Mo). Sur Kaggle : Save Version > Save & Run All, puis onglet Output." % (Path("emotion_camera_bundle.zip").stat().st_size / 1e6))

# Conclusion (à rédiger avec vos résultats)

Appuyez-vous sur les tableaux ci-dessus, pas seulement sur le score le plus élevé :

- **Données** : nombre d'images après suppression des copies, nombre de sujets, déséquilibre des classes.
- **Meilleur HOG** : …  **Meilleur LBP** : …
- **Fusion HOG+LBP** : apport ou non (comparer à l'écart-type) : …
- **Landmarks** : mesures géométriques seules, patchs, fusion avec HOG/LBP : …
- **Sélection de caractéristiques** : effet sur les scores, la dimension et la vitesse : …
- **Modèle retenu pour la caméra** et pourquoi (performance, rapidité, simplicité) : …
- **Limites** : expressions jouées, peu de sujets, pas de classe « neutre », images 48×48.
- **Piste d'amélioration réaliste** : collecter quelques images « neutres » et spontanées pour la caméra ; réseau sur graphe de landmarks ; entraînement avec plus de variété d'éclairage et de poses.